### Transfer Learning

Transfer of learning occurs when an individual applies knowledge, skills, or habits learned in one context to a new or different situation. It is a core goal of education and training, ensuring that theoretical concepts are successfully utilized in real-world scenarios.


> **Transfer Learning with TensorFlow Part 1: Feature Extraction**

Transfer Learning is leveraging a working model's existing architecture and learned patterns gor our own problem.

There are two main benefits:
1. Can leverage and existing neural network architecture proven to work on similar to our own.
2. Can leverage and existing neural network architecture which has already learned the patterns on similardata to our own, then we can adapt those patterns to our own data.

In [1]:
# are we running the gpu
!nvidia-smi

/bin/bash: nvidia-smi: command not found


### Becoming one with data


In [2]:
# ## Loading the dataset
# import zipfile

# !curl -o ../../data/10_food_classes_all_data.zip https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_all_data.zip
# # unzip our data
# zip_ref = zipfile.ZipFile('../../data/10_food_classes_all_data.zip', 'r')
# zip_ref.extractall()
# zip_ref.close()

In [3]:
# walk through 10 classes of food images data
import os

for dirpath, dirname, filename in os.walk('convolutional network and computer vision/10_food_classes_all_data'):
    print(f'There are {len(dirname)} directories and {len(filename)} images in {dirpath}')

There are 2 directories and 0 images in convolutional network and computer vision/10_food_classes_all_data
There are 10 directories and 0 images in convolutional network and computer vision/10_food_classes_all_data/test
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/ice_cream
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/chicken_curry
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/steak
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/sushi
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/chicken_wings
There are 0 directories and 250 images in convolutional network and computer vision/10_food_classes_all_data/test/grilled_salmon
There are 0 directories and 250

## Craete data loaders (preparing the data)

We'll use the `ImageDataGenerator` class to load in our images in batches.

In [5]:
# Setup data input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224, 224)
BATCH_SIZE = 32

train_dir = 'convolutional network and computer vision/10_food_classes_all_data/train/'
test_dir = 'convolutional network and computer vision/10_food_classes_all_data/test/'

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen = ImageDataGenerator(rescale=1/255.)

print('Training images:')
train_data_10_percent = train_datagen.flow_from_directory(train_dir,
                                                          target_size=IMAGE_SHAPE,
                                                          batch_size=BATCH_SIZE,
                                                          class_mode='categorical')

print('Testing images:')
test_data = test_datagen.flow_from_directory(test_dir,
                                             target_size=IMAGE_SHAPE,
                                             batch_size=BATCH_SIZE,
                                             class_mode='categorical')


Training images:
Found 7500 images belonging to 10 classes.
Testing images:
Found 2500 images belonging to 10 classes.


### Setting up callabcks (things to run whilst our model trains)

Callbacks - utilities called at certain point during model traning.

Callbacks are extra functionality you can add to your models to be performed during or after training. Some of the most popular callbacks:

* Tracking Experiments with the TensorBoard callback
* Model checkpoint with the ModelCheckpoint callback
* Stopping a model from training (before it trains too long and overfits) with the EarlyStopping callback


In [8]:
# tensorboard callback
import datetime
import tensorflow as tf

def create_tensorboard_callback(dir_name, experiment_name):
    log_dir = dir_name + '/' + experiment_name + '/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    tensorboard_callbacks = tf.keras.callbacks.TensorBoard(log_dir=log_dir)
    print(f'Saving tensorboard log files to: {log_dir}')
    return tensorboard_callbacks


### Creating model using TensorFlow Hub

In the past we've used TensorFlow to create our own models layer by layer from scratch.

Now we're going to do a similar process, except the majority of our model's layer are going to come from TensorFlow Hub.

We can access pretrained models on: https://tfhub.dev/

In [14]:
# lets compare the following two models
resnet_url = "https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/5"

efficientnet_url = "https://tfhub.dev/tensorflow/efficientnet/b0/feature-vector/1"

In [15]:
# Import dependencies
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers

In [17]:
# Let's make a create model function
def create_model(model_url, num_classes=10):
    """
    Creates a TensorFlow Keras model using a pretrained feature extractor
    from TensorFlow Hub and adds a custom output layer.

    Args:
        model_url (str): URL of the TensorFlow Hub feature extraction model.
        num_classes (int): Number of output classes for classification.
                           Default is 10.

    Returns:
        tf.keras.Model: A compiled Keras Sequential model consisting of
                        a frozen feature extraction layer and a Dense
                        output layer with softmax activation.
    """

    feature_extractor_layer = hub.KerasLayer(
        model_url,
        trainable=False,
        name="feature_extraction_layer",
        input_shape=IMAGE_SHAPE+(3,))

    model = tf.keras.Sequential([
        feature_extractor_layer,
        layers.Dense(num_classes, 
                     activation="softmax",
                     name='Output Layer')
    ])

    return model

In [20]:
## new way
from tensorflow.keras import layers, models

def create_model(num_classes=10):
    """
    Creates a native Keras 3 model using ResNet50 as a feature extractor.
    """
    # 1. Load a native pretrained ResNet50 base model
    base_model = tf.keras.applications.ResNet50(
        include_top=False,     # Removes the original 1000-class classification head
        weights='imagenet',    # Uses pretrained ImageNet weights
        input_shape=IMAGE_SHAPE+(3,)
    )

    # 2. Freeze the feature extractor weights
    base_model.trainable = False

    # 3. Stack them cleanly in a standard Sequential pipeline
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(), # Flattens the ResNet 4D output to 2D for the Dense layer
        layers.Dense(num_classes, 
                     activation="softmax",
                     name='Output_Layer')
    ])

    return model

### Creating and testing ResNet TensorFlow Hub  Feature Extraction model

In [21]:
# # create resnet model
# resnet_model = create_model(resnet_url,
#                             num_classes=train_data_10_percent.num_classes)

resnet_model = create_model(num_classes=train_data_10_percent.num_classes)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 20s 0us/step


In [22]:
resnet_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │        20,490 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,608,202 (90.06 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [23]:
# compile our resnet model
resnet_model.compile(loss='categorical_crossentropy',
                     optimizer=tf.keras.optimizers.Adam(),
                     metrics=['accuracy'])

In [ ]:
# fit the model
# Fit the model
resnet_history = resnet_model.fit(
                                train_data_10_percent,
                                epochs=2, # 5 epochs is a great starting point for transfer learning
                                steps_per_epoch=len(train_data_10_percent),
                                validation_data=test_data, # replace with your validation/test variable name if different
                                callbacks=[create_tensorboard_callback(dir_name='../tensorflow_hub',
                                                                       experiment_name='resnet50V2')]
)

Saving tensorboard log files to: ../tensorflow_hub/resnet50V2/20260531-181211
Epoch 1/2
235/235 ━━━━━━━━━━━━━━━━━━━━ 906s 4s/step - accuracy: 0.1751 - loss: 2.2379 - val_accuracy: 0.1796 - val_loss: 2.2229
Epoch 2/2
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1748 - loss: 2.2199